# 🕵️ 03 · The Leakage Hunt — 0.955 → 0.838

> **Chapter 3.** The story the whole project is built on. Three
> distinct leakage sources were auditioned and removed. The PR-AUC
> dropped, and that is the point — a lower honest number is worth
> more than a higher contaminated one.

## The five-stage decline

| Stage | PR-AUC | ROC-AUC | What was caught |
|---|---:|---:|---|
| 1. Initial baseline | 0.955 | 0.955 | Suspiciously flat ceiling. |
| 2. Missense filter | 0.819 | 0.934 | **64 %** of pathogenic rows had null `ref_aa` — non-missense leakage. |
| 3. Feature hygiene | 0.816 | 0.934 | `is_common` (100 % benign when True) and `chr` one-hot (15 % gain from dataset-composition artifact) removed. |
| 4. Paralog-aware split | 0.835 | 0.938 | Gene-level split shared 52 % of gene-prefix families between train and test. Family-level split fixes it. |
| 5. Optuna retuning | 0.836 | 0.938 | 40 TPE trials landed within 0.001 of each other → at the feature-limited ceiling. |

The same table is the headline of `docs/CHANGELOG.md` (0.2.0
release). Below, we reload the committed journey CSV so the figure
reflects the latest numbers byte-for-byte.

---

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path as _Path
sys.path.insert(0, str(_Path.cwd().parent))
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

REPO = Path.cwd().parent
journey = pd.read_csv(REPO / "results/metrics/leakage_fix_journey.csv")
print(journey.to_string(index=False))

In [ ]:
# Re-render the journey figure from the committed CSV.
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(journey))
ax.plot(x, journey["test_pr_auc"], "o-", label="PR-AUC", color="#e74c3c", lw=2)
ax.plot(x, journey["test_roc_auc"], "s-", label="ROC-AUC", color="#3498db", lw=2)
ax.set_xticks(x)
ax.set_xticklabels(journey["stage"].astype(str).tolist(), rotation=20, ha="right")
ax.set_ylabel("Metric value")
ax.set_ylim(0.78, 0.98)
ax.legend(loc="lower left", frameon=False)
ax.set_title("Leakage-fix journey — 5-stage decline to an honest baseline",
             fontweight="bold", loc="left")
fig.tight_layout()
plt.show()

## Leakage source 1 — the non-missense contamination

Pre-audit the pipeline carried rows with missing `ref_aa` OR `alt_aa`
(i.e. variants that are not missense — they're stop-gained, splice,
synonymous, UTR, etc.). Those rows had null AA-derived features
*and* were disproportionately pathogenic. The model picked up the
correlation "no AA features → pathogenic" as a shortcut.

The drop from 0.955 → 0.819 PR-AUC is the biggest single fix in
the project.

In [ ]:
# Confirm the fix holds: zero non-missense rows in the committed train split.
train = pd.read_parquet(REPO / "data/splits/train.parquet")
non_missense = int((train["ref_aa"].isna() | train["alt_aa"].isna()).sum())
assert non_missense == 0, f"{non_missense} non-missense rows leaked in!"
print(f"✅ training split: {len(train):,} rows, 0 non-missense")

## Leakage source 2 — feature hygiene

Two features were banned after the audit:

1. **`is_common`** — defined as *"AF_popmax > 0.01"*. By ClinVar
   convention, any variant labeled `is_common=True` was **100 % benign**.
   Including this feature is definitional circularity. The XGBoost
   model assigned it 0 % gain incidentally, but we removed it for
   academic hygiene anyway.

2. **`chr` one-hot** — the chromosome the variant is on. This
   correlates with dataset composition (chr17 over-represents BRCA1
   pathogenic variants, chrX DMD etc.) rather than biology. Removed.

The banlist is enforced by `src/verify_no_leakage.py`, which is wired
into CI through `tests/test_verify_no_leakage.py`.

In [ ]:
# Verify the banlist on the current feature columns.
BANNED = {"is_common", "chr", "ref", "alt"}
cols = pd.read_csv(REPO / "results/metrics/xgboost_feature_columns.csv")
names = set(cols.iloc[:, 0].astype(str))
contaminated = names & BANNED
onehot_banned = [c for c in names if c.startswith(("chr_", "is_common_"))]
assert not contaminated, f"banned features in feature matrix: {contaminated}"
assert not onehot_banned, f"banned one-hot cols: {onehot_banned[:5]}"
print("✅ no banned features in the current training matrix")

## Leakage source 3 — paralog leakage (subtle)

A plain gene-level train/val/test split seems safe — "the test gene
is never seen during training". But **gene families are highly
redundant**: KRT1, KRT5, KRT14 all share >80 % sequence identity. If
KRT1 is in train and KRT14 is in test, the model has effectively
"seen" most of KRT14 through KRT1.

The fix: **family-level splits**. `src.data_splitting.assign_gene_family`
uses 16 regex patterns to collapse large paralog clusters (keratins,
cadherins, zinc-fingers, HLA, olfactory receptors, etc.) and a
trailing-digit fallback for the long tail. The result is **7,851
families** from **15,479 genes**.

The committed train/val/test are regression-tested to have **zero
shared families** (see `tests/test_data_splitting.py`).

In [ ]:
import sys
sys.path.insert(0, str((Path.cwd().parent).resolve()))
from src.data_splitting import assign_gene_family

train = pd.read_parquet(REPO / "data/splits/train.parquet")
val = pd.read_parquet(REPO / "data/splits/val.parquet")
test = pd.read_parquet(REPO / "data/splits/test.parquet")

train_fams = {assign_gene_family(g) for g in train["gene"].unique()}
val_fams = {assign_gene_family(g) for g in val["gene"].unique()}
test_fams = {assign_gene_family(g) for g in test["gene"].unique()}

print(f"Family counts:")
print(f"  train  {len(train_fams):>5,}")
print(f"  val    {len(val_fams):>5,}")
print(f"  test   {len(test_fams):>5,}")
print(f"  total  {len(train_fams | val_fams | test_fams):>5,}")
print()
print(f"Overlaps (must all be zero):")
print(f"  train ∩ val    {len(train_fams & val_fams)}")
print(f"  train ∩ test   {len(train_fams & test_fams)}")
print(f"  val ∩ test     {len(val_fams & test_fams)}")

## The post-audit picture

At stage 5 the baseline stopped improving no matter how much Optuna
compute we threw at it (40 trials landed within 0.001 of each other).
That's a strong signal the **model is at its feature-limited
ceiling**: the information in the 33 tabular features genuinely
caps performance at PR-AUC ≈ 0.836.

Post-audit, Phase 2 step 1 added 5 gnomAD gene-level constraint
features and bumped the test PR-AUC to 0.838. That micro-improvement
is the value of gene-level priors on the *in-distribution* task; the
real payoff shows up on **external denovo-db generalization** (see
notebook 07).

---

## Reproduce

```bash
# The leakage audit was a one-time pipeline rebuild; the journey CSV
# is hand-audited from experiment logs. To confirm the current state
# still passes every gate:
python -m src.verify_no_leakage
pytest tests/test_verify_no_leakage.py tests/test_data_splitting.py
```